In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('Adiabatic_computing.pdf') # loading the document for knowledge base construction.
docs = loader.load()

C:\Users\Rishabh Debnath\AppData\Local\Temp\ipykernel_21040\175853881.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_classic.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    embedding_function=embedding_model,
    persist_directory='RAG_vectorstore_test_db',
    collection_name='research_papers'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2332.76it/s]
C:\Users\Rishabh Debnath\AppData\Local\Temp\ipykernel_21040\3718141447.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20              # for the moment using recursive character text splitting, later would move on to semantic chunking.
)

In [6]:
chunks = splitter.split_documents(docs)

In [7]:
chunks[0].metadata

{'producer': 'RealObjects PDFreactor(R) 10.2.10722, Serial No: 3892, Licensed for: Oxford University Press',
 'creator': 'RealObjects PDFreactor(R) 10.2.10722, Serial No: 3892, Licensed for: Oxford University Press',
 'creationdate': '2022-05-09T07:59:12+00:00',
 'moddate': '2022-05-09T07:59:12+00:00',
 'subject': 'Adiabatic quantum computing (AQC) is a model of computation that uses quantum mechanical processes operating under adiabatic conditions. As a form of universal quantum computation, AQC employs the principles of superposition, tunneling, and entanglement that manifest in quantum physical systems. The AQC model of quantum computing is distinguished by the use of dynamical evolution that is slow with respect to the time and energy scales of the underlying physical systems. This adiabatic condition enforces the promise that the quantum computational state will remain well-defined and controllable thus enabling the development of new algorithmic approaches.Several notable algorit

In [8]:
vectorstore.add_documents(chunks)

['ea2075ed-eb3d-44c4-abf5-5c9eeafe3416',
 'b20a93f3-1ba5-4a1d-9b40-94944b67ec69',
 '187adfc2-dd68-4f4b-b514-159e2d08a1a5',
 '01985186-5b37-4647-8135-0952d25c77e2',
 '1df41429-98a3-42e2-a24f-dfdfe7c16539',
 '1e27c86c-9157-4bf6-8819-2683edc653aa',
 '3df0ea03-5b08-4d95-bd04-a25204115bbf',
 'd6fb64ed-b9dc-4f1b-8001-143c75c4ed52',
 '32d10949-9477-42f7-93b5-796e7c636e44',
 '096f63fd-2a03-4ee4-9d24-645cce60fb55',
 '74376d67-52ba-4144-898c-242c5707b619',
 'eb940f09-eaea-40f6-b2cb-488461f18d7d',
 '6d7ba0f3-9ddd-4d2e-ba69-09e9955cce1b',
 '044ebf29-d0bb-4274-978f-d0ce38fb60ce',
 '30cc0151-ab4d-430e-b6b7-efa83f9f8b0b',
 'e3cec37a-fda5-426a-8e97-6abe27dd1afe',
 '8d08aa80-3dee-4938-b48e-0e3dd3cd0e1e',
 '4ca0bc8d-ae52-4753-b6b1-791783ee8396',
 'd1f5bebd-7969-471f-a85c-212c31ca8970',
 '1044f771-d1d8-4e0e-8197-e7e1134495c7',
 '18a6b02e-a44a-40a5-b55b-1bae574f6ab9',
 '831beab6-eb33-4748-a818-2958df82c66a',
 '4fe5e693-c35b-48e0-a4b2-71c5d324793d',
 'f79ecb4a-55e2-47e5-ab6f-0932f0a471b1',
 'ab5b3148-8cbe-

In [10]:
from langchain_groq import ChatGroq
from langchain_classic.retrievers import MultiQueryRetriever

llm = ChatGroq(model='llama-3.3-70b-versatile')
retrieval = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k':10}, search_type='similarity'),
    llm=llm
)

In [17]:
query = 'What is Adiabatic Computing?'

results = retrieval.invoke(query)
for i, docs in enumerate(results):
    print(f'--Result {i+1}-- \n {docs.page_content} \n')

--Result 1-- 
 have been developed to implement universal quantum computation with each being 
computationally equivalent but operationally distinct. The model of adiabatic quantum 
computing is described in detail below, while a summary of the prominent circuit model can be 
found elsewhere (Nielsen & Chuang, 2010). The choice of quantum computing model may be 
tailored to the intended purpose of a quantum computer as some models more efficiently express 
certain algorithms. In addition, limits on the controllability of a quantum physical system may 
also make certain computational models better suited.
2.  Adiabatic Quantum Computing
Adiabatic quantum computing (AQC) is a model of computation that uses quantum-mechanical 
processes operating under adiabatic conditions. This model employs continuous-time evolution 
of a quantum state  from a well-defined initial value to compute a final observed value. The 
evolution is modeled by the Schrödinger equation 

--Result 2-- 
 Adiabatic Qu

In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

parser = StrOutputParser()

prompt = PromptTemplate(
    template='You are a helpful assistant, following is provided to you a context on which a question-> {question} is asked. Kindly answer the question from only the following context, if the context is insufficient then simply say you have inadequate information to answer the question. \n context -> {context}',
    input_variables=['question', 'context']
)

parallel_chain = RunnableParallel({
    'context': retrieval | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

chain = parallel_chain | prompt | llm | parser
result = chain.invoke(query)
print(result)

Adiabatic Computing is not explicitly defined in the provided context. However, Adiabatic Quantum Computing (AQC) is described as a model of computation that uses quantum-mechanical processes operating under adiabatic conditions, employing continuous-time evolution of a quantum state to compute a final observed value. If "Adiabatic Computing" refers to the same concept as Adiabatic Quantum Computing, then it can be described as a model of computation that uses quantum-mechanical processes. But without more information, it's unclear if "Adiabatic Computing" is being used as a distinct term.
